# NeurCross Metadata Dashboard

This notebook builds a dashboard from the GitHub metadata repository for NeurCross training runs.

It reads from a configurable GitHub repo and branch, then loads:
- `metadata/training_tracker.jsonl`
- `metadata/training_state.json`
- `metadata/claims/*.json`

The dashboard focuses on:
- run status and progress
- dataset/source coverage
- GPU/device usage
- failed/skipped/uploaded runs
- claim activity
- simplification activity
- resumable failed-run inventory

The notebook is designed to run locally or on Google Colab.


In [ ]:
from __future__ import annotations

import base64
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import Markdown, display

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

GITHUB_METADATA_REPO = "akashskypatel/NeurCross-Metadata-TEST"
GITHUB_METADATA_BRANCH = "main"
GITHUB_TOKEN_ENV = "GH_TOKEN"
REQUEST_TIMEOUT_SECONDS = 60

def running_on_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ

def running_on_kaggle() -> bool:
    return "KAGGLE_KERNEL_RUN_TYPE" in os.environ

def get_secret(name: str, default: str | None = None) -> str | None:
    value = os.environ.get(name)
    if value:
        return value
    if running_on_colab():
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    if running_on_kaggle():
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name)
            if value:
                return value
        except Exception:
            pass
    return default

GH_TOKEN = get_secret(GITHUB_TOKEN_ENV)
if GH_TOKEN:
    GH_TOKEN = GH_TOKEN.strip()

print(f"GitHub metadata repo: {GITHUB_METADATA_REPO}")
print(f"GitHub metadata branch: {GITHUB_METADATA_BRANCH}")
print("GitHub token detected." if GH_TOKEN else "GitHub token not detected; using anonymous GitHub access.")


In [ ]:
def github_api_headers() -> dict:
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    if GH_TOKEN:
        headers["Authorization"] = f"Bearer {GH_TOKEN}"
    return headers

def github_api_get_json(url: str, *, params: dict | None = None):
    response = requests.get(url, headers=github_api_headers(), params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    return response.json()

def github_repo_contents_url(path: str) -> str:
    normalized = path.strip("/")
    return f"https://api.github.com/repos/{GITHUB_METADATA_REPO}/contents/{normalized}"

def github_read_text(path: str) -> str:
    payload = github_api_get_json(github_repo_contents_url(path), params={"ref": GITHUB_METADATA_BRANCH})
    if isinstance(payload, dict) and payload.get("encoding") == "base64":
        return base64.b64decode(payload["content"]).decode("utf-8")
    download_url = payload.get("download_url") if isinstance(payload, dict) else None
    if not download_url:
        raise RuntimeError(f"Could not resolve text payload for {path}")
    response = requests.get(download_url, headers=github_api_headers(), timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    return response.text

def github_list_claim_paths() -> list[str]:
    payload = github_api_get_json(github_repo_contents_url("metadata/claims"), params={"ref": GITHUB_METADATA_BRANCH})
    if not isinstance(payload, list):
        return []
    claim_paths = []
    for item in payload:
        if item.get("type") == "file" and str(item.get("name", "")).endswith(".json"):
            claim_paths.append(item["path"])
    return sorted(claim_paths)

def _read_json_text(text: str) -> dict:
    return json.loads(text)

def _read_jsonl_text(text: str) -> list[dict]:
    records = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        records.append(json.loads(line))
    return records

def _read_claims_from_github() -> list[dict]:
    claims = []
    for claim_path in github_list_claim_paths():
        payload = _read_json_text(github_read_text(claim_path))
        payload["claim_file"] = Path(claim_path).name
        claims.append(payload)
    return claims

def _normalize_optional_text(value) -> str:
    if value is None:
        return ''
    if isinstance(value, float) and pd.isna(value):
        return ''
    if pd.isna(value):
        return ''
    return str(value).strip().lower()

def classify_failure_reason(status: str | None, error_text: str | None) -> str | None:
    status_value = _normalize_optional_text(status)
    text = _normalize_optional_text(error_text)
    if not status_value:
        return None
    if status_value == 'completed':
        return None
    if status_value == 'started':
        return 'in_progress'
    if status_value == 'skipped':
        if 'exceeds dynamic gpu face limit' in text:
            return 'gpu_face_limit_exceeded'
        if 'simplification failed' in text:
            return 'mesh_simplification_failed'
        if 'preflight' in text:
            return 'preflight_rejected'
        return 'skipped_other'
    if status_value == 'failed':
        if 'outofmemoryerror' in text or 'cuda out of memory' in text:
            return 'cuda_oom'
        if 'training_step_failed' in text:
            return 'training_step_failed'
        if 'training_loop_failed' in text:
            return 'training_loop_failed'
        if 'training_finalize_failed' in text:
            return 'training_finalize_failed'
        if 'dataset-label setup failed' in text or 'setup failed' in text:
            return 'training_setup_failed'
        if 'nonetype' in text or 'subscriptable' in text or 'traceback' in text:
            return 'training_code_error'
        if 'load_checkpoint' in text or 'resume' in text:
            return 'resume_failed'
        return 'failed_other'
    return status_value

def _to_frame(records: list[dict]) -> pd.DataFrame:
    if not records:
        return pd.DataFrame()
    df = pd.json_normalize(records)
    for col in df.columns:
        if col.endswith("_at_utc") or col in {"updated_at_utc", "expires_at_utc", "released_at_utc"}:
            try:
                df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
            except Exception:
                pass
    return df

def load_metadata_frames() -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    state = _read_json_text(github_read_text("metadata/training_state.json"))
    tracker_df = _to_frame(_read_jsonl_text(github_read_text("metadata/training_tracker.jsonl")))
    claims_df = _to_frame(_read_claims_from_github())

    if not tracker_df.empty:
        if "uploaded_to_hf" in tracker_df:
            tracker_df["uploaded_to_hf"] = tracker_df["uploaded_to_hf"].fillna(False).astype(bool)
        else:
            tracker_df["uploaded_to_hf"] = False
        if "simplification_applied" in tracker_df:
            tracker_df["simplification_applied"] = tracker_df["simplification_applied"].fillna(False).astype(bool)
        else:
            tracker_df["simplification_applied"] = False
        tracker_df["has_resume_checkpoint"] = tracker_df.get("resume_checkpoint_path").notna() if "resume_checkpoint_path" in tracker_df else False
        tracker_df["is_failed"] = tracker_df.get("status").eq("failed")
        tracker_df["is_skipped"] = tracker_df.get("status").eq("skipped")
        tracker_df["is_started"] = tracker_df.get("status").eq("started")
        tracker_df["is_completed"] = tracker_df.get("status").eq("completed")
        tracker_df["failure_reason_group"] = [
            classify_failure_reason(status, error)
            for status, error in zip(tracker_df.get("status", pd.Series(dtype=object)), tracker_df.get("error", pd.Series(dtype=object)))
        ]

    if not claims_df.empty:
        claims_df["is_active_claim"] = claims_df.get("status").eq("claimed")
        claims_df["is_released_claim"] = claims_df.get("status").eq("released")

    return state, tracker_df, claims_df

state, tracker_df, claims_df = load_metadata_frames()
print(f"Tracker records: {len(tracker_df):,}")
print(f"Claim files: {len(claims_df):,}")
print("State snapshot:")
display(pd.DataFrame([state]))


In [ ]:
summary_rows = []
summary_rows.append({'metric': 'tracker_records', 'value': int(len(tracker_df))})
summary_rows.append({'metric': 'claim_files', 'value': int(len(claims_df))})
summary_rows.append({'metric': 'runs_completed_state', 'value': int(state.get('runs_completed', 0))})
if not tracker_df.empty:
    summary_rows.extend([
        {'metric': 'completed_records', 'value': int((tracker_df['status'] == 'completed').sum())},
        {'metric': 'failed_records', 'value': int((tracker_df['status'] == 'failed').sum())},
        {'metric': 'skipped_records', 'value': int((tracker_df['status'] == 'skipped').sum())},
        {'metric': 'started_records', 'value': int((tracker_df['status'] == 'started').sum())},
        {'metric': 'uploaded_to_hf', 'value': int(tracker_df['uploaded_to_hf'].sum())},
        {'metric': 'simplified_for_training', 'value': int(tracker_df['simplification_applied'].sum()) if 'simplification_applied' in tracker_df else 0},
        {'metric': 'resumable_failed_runs', 'value': int(((tracker_df['status'] == 'failed') & tracker_df['resume_checkpoint_path'].notna()).sum()) if 'resume_checkpoint_path' in tracker_df else 0},
    ])
if not claims_df.empty:
    summary_rows.extend([
        {'metric': 'active_claims', 'value': int((claims_df['status'] == 'claimed').sum())},
        {'metric': 'released_claims', 'value': int((claims_df['status'] == 'released').sum())},
    ])
summary_df = pd.DataFrame(summary_rows)
display(summary_df)


In [ ]:
if tracker_df.empty:
    raise RuntimeError('No tracker records found.')

status_counts = tracker_df['status'].fillna('unknown').value_counts().sort_index()
device_counts = tracker_df['training_device'].fillna('unknown').value_counts().sort_index() if 'training_device' in tracker_df else pd.Series(dtype='int64')
dataset_counts = tracker_df['dataset_id'].fillna('unknown').value_counts().head(15) if 'dataset_id' in tracker_df else pd.Series(dtype='int64')

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
status_counts.plot(kind='bar', ax=axes[0], title='Run Status Counts', color='#4C78A8')
axes[0].set_ylabel('records')
device_counts.plot(kind='bar', ax=axes[1], title='Training Device Counts', color='#F58518')
axes[1].set_ylabel('records')
dataset_counts.sort_values().plot(kind='barh', ax=axes[2], title='Top Dataset Counts', color='#54A24B')
axes[2].set_xlabel('records')
plt.tight_layout()
plt.show()


In [ ]:
if 'updated_at_utc' in tracker_df and tracker_df['updated_at_utc'].notna().any():
    timeline = tracker_df.copy()
    timeline['updated_day'] = timeline['updated_at_utc'].dt.floor('D')
    status_by_day = timeline.groupby(['updated_day', 'status']).size().unstack(fill_value=0).sort_index()
    status_by_day_plot = status_by_day.copy()
    status_by_day_plot.index = status_by_day_plot.index.strftime('%Y-%m-%d')
    display(status_by_day.tail(14))
    ax = status_by_day_plot.plot(kind='bar', stacked=True, figsize=(14, 6), title='Status Updates by Day')
    ax.set_ylabel('records')
    plt.tight_layout()
    plt.show()


In [ ]:
interesting_cols = [
    'status', 'dataset_id', 'mesh_relative_path', 'sample_id', 'training_device', 'attempt',
    'uploaded_to_hf', 'hf_path', 'recommended_destination', 'resume_checkpoint_path',
    'face_count', 'training_face_count', 'gpu_face_limit', 'simplification_applied',
    'simplification_target_face_count', 'simplification_error', 'error', 'updated_at_utc'
]
present_cols = [col for col in interesting_cols if col in tracker_df.columns]
failed_df = tracker_df[tracker_df['status'].isin(['failed', 'skipped'])].copy()
failed_df = failed_df.sort_values('updated_at_utc', ascending=False) if 'updated_at_utc' in failed_df else failed_df
display(failed_df[present_cols].head(50))


In [ ]:
failure_df = tracker_df[tracker_df['status'].isin(['failed', 'skipped'])].copy()
if not failure_df.empty and 'failure_reason_group' in failure_df.columns:
    reason_counts = failure_df['failure_reason_group'].fillna('unknown').value_counts().sort_values(ascending=True)
    display(reason_counts.to_frame('count'))
    ax = reason_counts.plot(kind='barh', figsize=(10, 5), title='Failure Count by Generalized Reason', color='#E45756')
    ax.set_xlabel('records')
    ax.set_ylabel('reason')
    plt.tight_layout()
    plt.show()


In [ ]:
if not claims_df.empty:
    claim_cols = [col for col in ['status', 'dataset_id', 'mesh_relative_path', 'sample_id', 'owner_id', 'expires_at_utc', 'released_at_utc', 'final_status', 'updated_at_utc', 'claim_file'] if col in claims_df.columns]
    display(claims_df[claim_cols].sort_values('updated_at_utc', ascending=False).head(50) if 'updated_at_utc' in claims_df else claims_df[claim_cols].head(50))

    claim_status = claims_df['status'].fillna('unknown').value_counts().sort_index()
    ax = claim_status.plot(kind='bar', figsize=(8, 4), title='Claim Status Counts', color='#B279A2')
    ax.set_ylabel('claims')
    plt.tight_layout()
    plt.show()


In [ ]:
if 'simplification_applied' in tracker_df.columns:
    simpl_df = tracker_df.copy()
    simpl_counts = simpl_df['simplification_applied'].fillna(False).value_counts().rename(index={True: 'simplified', False: 'not_simplified'})
    display(simpl_counts.to_frame('count'))
    ax = simpl_counts.plot(kind='bar', figsize=(8, 4), title='Simplification Usage', color='#E45756')
    ax.set_ylabel('records')
    plt.tight_layout()
    plt.show()

    simpl_cols = [col for col in ['dataset_id', 'mesh_relative_path', 'training_device', 'face_count', 'training_face_count', 'gpu_face_limit', 'simplification_target_face_count', 'simplification_error', 'status', 'updated_at_utc'] if col in simpl_df.columns]
    display(simpl_df[simpl_df['simplification_applied'] == True][simpl_cols].sort_values('updated_at_utc', ascending=False).head(50) if 'updated_at_utc' in simpl_df else simpl_df[simpl_df['simplification_applied'] == True][simpl_cols].head(50))


In [ ]:
if widgets is None:
    display(Markdown('`ipywidgets` is not available in this environment; skipping interactive filter view.'))
else:
    dataset_options = ['<all>'] + sorted(str(value) for value in tracker_df['dataset_id'].dropna().unique()) if 'dataset_id' in tracker_df else ['<all>']
    status_options = ['<all>'] + sorted(str(value) for value in tracker_df['status'].dropna().unique()) if 'status' in tracker_df else ['<all>']
    device_options = ['<all>'] + sorted(str(value) for value in tracker_df['training_device'].dropna().unique()) if 'training_device' in tracker_df else ['<all>']

    dataset_dd = widgets.Dropdown(options=dataset_options, description='dataset')
    status_dd = widgets.Dropdown(options=status_options, description='status')
    device_dd = widgets.Dropdown(options=device_options, description='device')
    limit_slider = widgets.IntSlider(value=25, min=5, max=200, step=5, description='rows')

    def filtered_view(dataset_value, status_value, device_value, row_limit):
        df = tracker_df.copy()
        if dataset_value != '<all>' and 'dataset_id' in df:
            df = df[df['dataset_id'] == dataset_value]
        if status_value != '<all>' and 'status' in df:
            df = df[df['status'] == status_value]
        if device_value != '<all>' and 'training_device' in df:
            df = df[df['training_device'] == device_value]
        if 'updated_at_utc' in df:
            df = df.sort_values('updated_at_utc', ascending=False)
        cols = [col for col in [
            'status', 'dataset_id', 'mesh_relative_path', 'sample_id', 'training_device', 'attempt',
            'uploaded_to_hf', 'recommended_destination', 'resume_checkpoint_path', 'hf_path',
            'face_count', 'training_face_count', 'gpu_face_limit', 'simplification_applied',
            'error', 'updated_at_utc'
        ] if col in df.columns]
        display(df[cols].head(row_limit))

    display(widgets.interactive_output(filtered_view, {
        'dataset_value': dataset_dd,
        'status_value': status_dd,
        'device_value': device_dd,
        'row_limit': limit_slider,
    }))
    display(widgets.HBox([dataset_dd, status_dd, device_dd, limit_slider]))
